In [ ]:
from pyspark.sql.functions import current_timestamp

# =========================================================
# 10_LANDING_CUSTOMERS
# Simple incremental load
# No audit
# No watermark
# =========================================================

source_path = "/Volumes/workspace/default/landing/customers*.csv"

# ---------------------------------------------------------
# 1. READ ALL CUSTOMER FILES
# ---------------------------------------------------------

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(source_path)
    .withColumn("load_timestamp", current_timestamp())
)

print(f"Rows read from source files: {df.count()}")

# ---------------------------------------------------------
# 2. CREATE LANDING TABLE IF IT DOES NOT EXIST
# ---------------------------------------------------------

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.landing.customers (
        customer_id STRING,
        first_name STRING,
        last_name STRING,
        email STRING,
        country STRING,
        city STRING,
        customer_type STRING,
        registration_date STRING,
        last_updated STRING,
        event_id STRING,
        batch_id STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
""")

# ---------------------------------------------------------
# 3. CREATE TEMP VIEW
# ---------------------------------------------------------

df.createOrReplaceTempView("customers_incoming")

# ---------------------------------------------------------
# 4. INSERT ONLY NEW EVENTS
# Incremental / idempotent load
# ---------------------------------------------------------

spark.sql("""
    INSERT INTO workspace.landing.customers
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.country,
        c.city,
        c.customer_type,
        c.registration_date,
        c.last_updated,
        c.event_id,
        c.batch_id,
        c.load_timestamp
    FROM customers_incoming c
    WHERE NOT EXISTS (
        SELECT 1
        FROM workspace.landing.customers l
        WHERE l.batch_id = c.batch_id
          AND l.event_id = c.event_id
    )
""")

# ---------------------------------------------------------
# 5. SHOW RESULT
# ---------------------------------------------------------

result = spark.sql("""
    SELECT
        batch_id,
        COUNT(*) AS rows_in_landing
    FROM workspace.landing.customers
    GROUP BY batch_id
    ORDER BY batch_id
""")

display(result)